# 04 — Neural Networks with PyTorch

## Learning Objectives

By the end of this notebook, you will understand:

- `torch.nn`
- `nn.Module`
- `nn.Parameter`
- `nn.Linear`
- `in_features`, `out_features`, `bias`
- Forward passes
- Weights and biases
- ReLU, Sigmoid, Tanh, Softmax
- Regression and classification losses
- `MSELoss`, `L1Loss`
- `CrossEntropyLoss`
- `BCELoss`, `BCEWithLogitsLoss`
- SGD, Momentum, Adam, AdamW
- Learning rate and weight decay
- The complete PyTorch training loop
- `train()` vs `eval()`
- Dropout and BatchNorm basics
- Validation and inference
- Saving/loading `state_dict`
- A complete classification project

# 1. From Autograd to Neural Networks

In the previous notebook we manually created:

```text
prediction = wx + b
loss
 ↓
backward()
 ↓
gradients
 ↓
parameter update
```

PyTorch provides higher-level tools so we do not need to manually manage every parameter.

The main building blocks are:

```text
nn.Module
   ↓
Layers
   ↓
Parameters
   ↓
Loss
   ↓
Optimizer
   ↓
Training loop
```

# 2. `torch.nn`

Import the neural-network module:

```python
import torch.nn as nn
```

It contains layers, activations, loss functions, containers, and other utilities for constructing neural networks.

In [1]:
import torch
import torch.nn as nn

print("PyTorch:", torch.__version__)

PyTorch: 2.9.1


# 3. `nn.Module`

Almost every PyTorch model is based on `nn.Module`.

Basic structure:

```python
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        ...
```

`__init__()` defines layers.

`forward()` defines how data flows through the model.

In [2]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x * 2

model = MyModel()

x = torch.tensor([1., 2., 3.])
print(model(x))

tensor([2., 4., 6.])


# 4. Why `super().__init__()`?

The parent `nn.Module` manages important PyTorch functionality such as:

- Parameters
- Child modules
- State dictionaries
- Device movement
- Training/evaluation mode
- Hooks

Therefore it is standard to call:

```python
super().__init__()
```

# 5. `nn.Linear`

A linear layer represents:

```text
y = xWᵀ + b
```

Syntax:

```python
nn.Linear(
    in_features,
    out_features,
    bias=True
)
```

### Parameters

- `in_features` → number of input features
- `out_features` → number of output features
- `bias` → whether a learnable bias is included

In [3]:
layer = nn.Linear(
    in_features=3,
    out_features=2
)

print(layer)

Linear(in_features=3, out_features=2, bias=True)


# 6. `in_features`

If each sample contains 3 values:

```text
[age, height, weight]
```

then:

```python
nn.Linear(3, ...)
```

is required.

For an input tensor:

```text
(batch_size, 3)
```

the last dimension must match `in_features`.

In [4]:
layer = nn.Linear(3, 2)

x = torch.randn(4, 3)
y = layer(x)

print("Input :", x.shape)
print("Output:", y.shape)

Input : torch.Size([4, 3])
Output: torch.Size([4, 2])


Shape flow:

```text
(batch, 3)
    ↓
Linear(3 → 2)
    ↓
(batch, 2)
```

The batch dimension is preserved.

# 7. `out_features`

`out_features` determines the number of values produced for each sample.

In [5]:
x = torch.randn(4, 3)

for out_features in [2, 4, 8]:
    layer = nn.Linear(3, out_features)
    y = layer(x)

    print(
        f"out_features={out_features} "
        f"→ output shape={tuple(y.shape)}"
    )

out_features=2 → output shape=(4, 2)
out_features=4 → output shape=(4, 4)
out_features=8 → output shape=(4, 8)


Changing `out_features` changes the output dimension and therefore changes the architecture of the network.

# 8. `bias`

With:

```python
bias=True
```

the layer computes:

```text
y = xWᵀ + b
```

With:

```python
bias=False
```

it computes:

```text
y = xWᵀ
```

No learnable bias vector is created.

In [6]:
with_bias = nn.Linear(3, 2, bias=True)
without_bias = nn.Linear(3, 2, bias=False)

print("With bias:")
for name, p in with_bias.named_parameters():
    print(name, p.shape)

print("\nWithout bias:")
for name, p in without_bias.named_parameters():
    print(name, p.shape)

With bias:
weight torch.Size([2, 3])
bias torch.Size([2])

Without bias:
weight torch.Size([2, 3])


# 9. Parameter Count of `nn.Linear`

For:

```python
nn.Linear(3, 2)
```

weights:

```text
3 × 2 = 6
```

bias:

```text
2
```

total:

```text
8 parameters
```

In [45]:
layer = nn.Linear(3, 2)

print("Weight:", layer.weight)
print("Weight shape:", layer.weight.shape)
print("Bias shape:", layer.bias.shape)
print("Weight count:", layer.weight.numel())
print("Bias count:", layer.bias.numel())
print("Total:", sum(p.numel() for p in layer.parameters()))

Weight: Parameter containing:
tensor([[ 0.5484,  0.1222, -0.4207],
        [ 0.5158, -0.2740, -0.2727]], requires_grad=True)
Weight shape: torch.Size([2, 3])
Bias shape: torch.Size([2])
Weight count: 6
Bias count: 2
Total: 8


# 10. Weights and Biases

PyTorch initializes the parameters automatically.

These values are learned during training.

In [8]:
layer = nn.Linear(3, 2)

print("Weights:")
print(layer.weight)

print("\nBias:")
print(layer.bias)

Weights:
Parameter containing:
tensor([[ 0.1488, -0.0057,  0.5297],
        [-0.3515, -0.4923,  0.2928]], requires_grad=True)

Bias:
Parameter containing:
tensor([ 0.2205, -0.4956], requires_grad=True)


# 11. `nn.Parameter`

Learnable model values are represented as `nn.Parameter`.

Let's inspect a layer's parameters.

In [9]:
layer = nn.Linear(3, 2)

for name, parameter in layer.named_parameters():
    print(
        name,
        "shape=", tuple(parameter.shape),
        "requires_grad=", parameter.requires_grad
    )

weight shape= (2, 3) requires_grad= True
bias shape= (2,) requires_grad= True


During training, autograd calculates:

```text
dLoss/dWeight
dLoss/dBias
```

and the optimizer uses those gradients to update the parameters.

# 12. Forward Pass

Calling:

```python
output = layer(x)
```

runs the layer.

For a complete model, the normal style is:

```python
output = model(x)
```

rather than calling `model.forward(x)` directly.

In [10]:
layer = nn.Linear(3, 2)

x = torch.tensor([
    [1., 2., 3.],
    [4., 5., 6.]
])

output = layer(x)

print("Input:")
print(x)

print("\nOutput:")
print(output)

Input:
tensor([[1., 2., 3.],
        [4., 5., 6.]])

Output:
tensor([[-1.6613, -0.6745],
        [-2.9235, -1.4137]], grad_fn=<AddmmBackward0>)


# 13. Activation Functions

A sequence of only linear layers can still be represented as a linear transformation.

Activation functions introduce non-linearity.

Common activations:

- ReLU
- Sigmoid
- Tanh
- Softmax

# 14. ReLU

Formula:

```text
ReLU(x) = max(0, x)
```

So:

```text
negative → 0
positive → unchanged
```

PyTorch:

```python
nn.ReLU()
```

In [11]:
relu = nn.ReLU()

x = torch.tensor([-3., -1., 0., 2., 5.])

print("Input:", x)
print("ReLU :", relu(x))

Input: tensor([-3., -1.,  0.,  2.,  5.])
ReLU : tensor([0., 0., 0., 2., 5.])


`nn.ReLU` has an optional:

```python
inplace=False
```

parameter.

For general teaching and safe default code, use:

```python
nn.ReLU()
```

unless an in-place operation is specifically needed.

In [12]:
print(nn.ReLU())
print(nn.ReLU(inplace=True))

ReLU()
ReLU(inplace=True)


# 15. Sigmoid

Formula:

```text
σ(x) = 1 / (1 + e⁻ˣ)
```

It maps values approximately into:

```text
(0, 1)
```

Commonly used for binary probability interpretation.

In [13]:
x = torch.tensor([-5., -2., 0., 2., 5.])

print(nn.Sigmoid()(x))

tensor([0.0067, 0.1192, 0.5000, 0.8808, 0.9933])


# 16. Tanh

Tanh maps values approximately into:

```text
(-1, 1)
```

In [14]:
x = torch.tensor([-5., -2., 0., 2., 5.])

print(nn.Tanh()(x))

tensor([-0.9999, -0.9640,  0.0000,  0.9640,  0.9999])


# 17. ReLU vs Sigmoid vs Tanh

Compare the same input:

In [15]:
x = torch.tensor([-3., -1., 0., 1., 3.])

print("x      :", x)
print("ReLU   :", nn.ReLU()(x))
print("Sigmoid:", nn.Sigmoid()(x))
print("Tanh   :", nn.Tanh()(x))

x      : tensor([-3., -1.,  0.,  1.,  3.])
ReLU   : tensor([0., 0., 0., 1., 3.])
Sigmoid: tensor([0.0474, 0.2689, 0.5000, 0.7311, 0.9526])
Tanh   : tensor([-0.9951, -0.7616,  0.0000,  0.7616,  0.9951])


| Activation | Approximate range | Common role |
|---|---|---|
| ReLU | `[0, ∞)` | Hidden layers |
| Sigmoid | `(0,1)` | Binary output interpretation |
| Tanh | `(-1,1)` | Some hidden/sequence models |

The correct activation depends on the architecture and task.

# 18. Softmax

Softmax converts a vector of scores into values that sum to approximately `1`.

```python
nn.Softmax(dim=...)
```

The `dim` parameter specifies the dimension containing the class scores.

In [16]:
scores = torch.tensor([
    [2.0, 1.0, 0.5],
    [1.0, 3.0, 0.0]
])

probabilities = nn.Softmax(dim=1)(scores)

print(probabilities)
print("Row sums:", probabilities.sum(dim=1))

tensor([[0.6285, 0.2312, 0.1402],
        [0.1142, 0.8438, 0.0420]])
Row sums: tensor([1.0000, 1.0000])


For:

```text
(batch_size, num_classes)
```

we normally use:

```python
dim=1
```

because dimension 1 contains the classes.

# 19. `nn.Sequential`

For simple feed-forward models, `nn.Sequential` can connect layers in order.

Example:

```text
4 inputs
 ↓
8 hidden neurons
 ↓
3 outputs
```

In [17]:
model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 3)
)

print(model)

x = torch.randn(5, 4)
y = model(x)

print("Input :", x.shape)
print("Output:", y.shape)

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=3, bias=True)
)
Input : torch.Size([5, 4])
Output: torch.Size([5, 3])


Shape flow:

```text
(5,4)
 ↓ Linear(4,8)
(5,8)
 ↓ ReLU
(5,8)
 ↓ Linear(8,3)
(5,3)
```

# 20. Custom `nn.Module`

Custom classes give us more control than `Sequential`.

In [18]:
class SimpleNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(4, 8)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(8, 3)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleNetwork()

print(model)

SimpleNetwork(
  (fc1): Linear(in_features=4, out_features=8, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=8, out_features=3, bias=True)
)


In [19]:
x = torch.randn(5, 4)
output = model(x)

print("Output shape:", output.shape)

Output shape: torch.Size([5, 3])


# 21. Inspecting Model Parameters

Use:

```python
model.parameters()
```

or:

```python
model.named_parameters()
```

In [20]:
for name, p in model.named_parameters():
    print(
        f"{name:15s} "
        f"shape={tuple(p.shape)} "
        f"count={p.numel()}"
    )

total = sum(p.numel() for p in model.parameters())
print("\nTotal parameters:", total)

fc1.weight      shape=(8, 4) count=32
fc1.bias        shape=(8,) count=8
fc2.weight      shape=(3, 8) count=24
fc2.bias        shape=(3,) count=3

Total parameters: 67


Parameter count is useful for understanding:

- Model size
- Memory requirements
- Computation
- Model capacity

# 22. Loss Functions

A loss function measures prediction error.

Common losses:

```text
MSELoss
L1Loss
CrossEntropyLoss
BCELoss
BCEWithLogitsLoss
```

The correct loss depends on the task and output representation.

# 23. `MSELoss`

Formula:

```text
MSE = mean((prediction - target)²)
```

Commonly used for regression.

In [ ]:
prediction = torch.tensor([2., 4., 6.])
target = torch.tensor([3., 5., 5.])

loss = nn.MSELoss()(prediction, target)

print("MSE:", loss.item())

MSE: 1.0


`MSELoss` supports:

```python
reduction="mean"
reduction="sum"
reduction="none"
```

The reduction controls how individual errors are combined.

In [22]:
prediction = torch.tensor([2., 4., 6.])
target = torch.tensor([3., 5., 5.])

for reduction in ["mean", "sum", "none"]:
    loss = nn.MSELoss(reduction=reduction)
    print(reduction, ":", loss(prediction, target))

mean : tensor(1.)
sum : tensor(3.)
none : tensor([1., 1., 1.])


# 24. L1 Loss

L1 loss uses absolute error:

```text
|prediction - target|
```

It is often less sensitive to large errors than squared error.

In [23]:
prediction = torch.tensor([2., 4., 10.])
target = torch.tensor([3., 5., 5.])

print("MSE:", nn.MSELoss()(prediction, target).item())
print("L1 :", nn.L1Loss()(prediction, target).item())

MSE: 9.0
L1 : 2.3333332538604736


# 25. Binary Classification

For binary classification:

```text
0 → negative
1 → positive
```

A model can output a single **logit**.

A sigmoid can convert a logit into a probability.

For training, PyTorch provides:

```python
nn.BCEWithLogitsLoss()
```

which combines sigmoid and binary cross entropy in a numerically stable implementation.

In [24]:
logits = torch.tensor([-2., 0., 2.])
targets = torch.tensor([0., 1., 1.])

loss = nn.BCEWithLogitsLoss()(logits, targets)

print("Loss:", loss.item())
print("Probabilities:", torch.sigmoid(logits))

Loss: 0.3156677782535553
Probabilities: tensor([0.1192, 0.5000, 0.8808])


### Important

With `BCEWithLogitsLoss`:

```text
model → raw logits → BCEWithLogitsLoss
```

Do **not** manually apply sigmoid before the loss.

# 26. `BCELoss`

`BCELoss` expects probabilities, normally in `[0,1]`.

So:

```text
logits
 ↓
sigmoid
 ↓
probability
 ↓
BCELoss
```

For standard binary classification, `BCEWithLogitsLoss` is generally the preferred combined form.

In [25]:
probabilities = torch.tensor([0.1, 0.8, 0.9])
targets = torch.tensor([0., 1., 1.])

loss = nn.BCELoss()(probabilities, targets)

print("BCE:", loss.item())

BCE: 0.14462153613567352


# 27. Multiclass Classification

Suppose there are 3 classes:

```text
0 = Cat
1 = Dog
2 = Car
```

The final layer produces 3 raw scores:

```text
[cat_score, dog_score, car_score]
```

For multiclass classification, use:

```python
nn.CrossEntropyLoss()
```

It expects **raw logits**.

In [26]:
logits = torch.tensor([
    [2.0, 0.5, 0.1],
    [0.2, 3.0, 0.4]
])

targets = torch.tensor([0, 1], dtype=torch.long)

loss = nn.CrossEntropyLoss()(logits, targets)

print("Loss:", loss.item())

Loss: 0.22174254059791565


Required shapes:

```text
logits  → (batch_size, num_classes)
targets → (batch_size,)
```

Targets contain class indices and normally use `torch.long`.

In [27]:
logits = torch.randn(4, 3)
targets = torch.tensor([0, 2, 1, 0], dtype=torch.long)

print("Logits :", logits.shape)
print("Targets:", targets.shape)
print("Dtype  :", targets.dtype)

loss = nn.CrossEntropyLoss()(logits, targets)
print("Loss:", loss.item())

Logits : torch.Size([4, 3])
Targets: torch.Size([4])
Dtype  : torch.int64
Loss: 0.9365334510803223


# 28. Why No Softmax Before `CrossEntropyLoss`?

Correct:

```python
loss = nn.CrossEntropyLoss()(logits, targets)
```

Do not normally do:

```python
probabilities = torch.softmax(logits, dim=1)
loss = nn.CrossEntropyLoss()(probabilities, targets)
```

`CrossEntropyLoss` expects logits and internally performs the appropriate log-softmax-related computation.

In [28]:
logits = torch.tensor([[2.0, 1.0, 0.5]])
target = torch.tensor([0])

correct = nn.CrossEntropyLoss()(logits, target)

probabilities = torch.softmax(logits, dim=1)
wrong_style = nn.CrossEntropyLoss()(probabilities, target)

print("Using logits:", correct.item())
print("After softmax:", wrong_style.item())

Using logits: 0.4643687903881073
After softmax: 0.826717734336853


# 29. Optimizers

The loss produces gradients.

The optimizer uses those gradients to update parameters.

Standard sequence:

```python
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

Common optimizers:

- SGD
- Adam
- AdamW

# 30. SGD

Basic syntax:

```python
torch.optim.SGD(
    model.parameters(),
    lr=0.01
)
```

Important parameters:

- `params`
- `lr`
- `momentum`
- `weight_decay`
- `nesterov`

In [29]:
model = nn.Linear(3, 1)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

print(optimizer)

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)


## Learning rate

The learning rate controls the approximate size of optimization steps.

```text
too small → slow learning
reasonable → stable progress
too large → unstable or divergent training
```

The best value depends on the model, data, optimizer, and training setup.

# 31. SGD with Momentum

Momentum adds a running direction to parameter updates.

Example:

```python
torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9
)
```

The exact behavior is determined by the optimizer implementation, but conceptually momentum helps smooth and accumulate update directions.

In [30]:
model = nn.Linear(3, 1)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9
)

print(optimizer)

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0.9
    nesterov: False
    weight_decay: 0
)


# 32. Weight Decay

`weight_decay` applies a regularization effect to parameter updates.

Example:

```python
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    weight_decay=1e-4
)
```

It is commonly used to discourage excessively large parameter values and can improve generalization in suitable problems.

# 33. Adam

Adam adapts updates using estimates of first and second moments of gradients.

Basic usage:

```python
torch.optim.Adam(
    model.parameters(),
    lr=0.001
)
```

Important parameters include:

- `lr`
- `betas`
- `eps`
- `weight_decay`

In [31]:
model = nn.Linear(3, 1)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print(optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


Typical Adam defaults are approximately:

```text
lr    = 0.001
betas = (0.9, 0.999)
eps   = 1e-8
```

The parameters control the optimizer's adaptive moment estimates and numerical stability.

# 34. AdamW

AdamW separates weight decay from the adaptive gradient update.

Basic usage:

```python
torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.01
)
```

AdamW is widely used in modern deep-learning training.

In [32]:
model = nn.Linear(3, 1)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.01
)

print(optimizer)

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.01
)


# 35. The Training Loop

A standard PyTorch training loop is:

```python
model.train()

for inputs, targets in dataloader:

    optimizer.zero_grad()

    outputs = model(inputs)

    loss = loss_fn(outputs, targets)

    loss.backward()

    optimizer.step()
```

Every line has a purpose.

### `model.train()`

Enables training behavior for modules such as Dropout and BatchNorm.

### `optimizer.zero_grad()`

Clears gradients accumulated from the previous batch.

### `model(inputs)`

Performs the forward pass.

### `loss_fn(...)`

Calculates the error.

### `loss.backward()`

Calculates gradients.

### `optimizer.step()`

Updates model parameters.

# 36. Complete Regression Network

Let's build:

```text
Input
 ↓
Linear(1 → 8)
 ↓
ReLU
 ↓
Linear(8 → 1)
 ↓
Prediction
```

Dataset:

```text
y = 2x + 1
```

In [33]:
torch.manual_seed(42)

x = torch.linspace(0, 10, 100).reshape(-1, 1)
y = 2 * x + 1

model = nn.Sequential(
    nn.Linear(1, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

loss_fn = nn.MSELoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

print(model)

Sequential(
  (0): Linear(in_features=1, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)


In [34]:
for epoch in range(1000):
    model.train()

    optimizer.zero_grad()

    prediction = model(x)

    loss = loss_fn(prediction, y)

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(
            f"Epoch {epoch+1:4d} | "
            f"Loss: {loss.item():.6f}"
        )

Epoch  100 | Loss: 28.594170
Epoch  200 | Loss: 7.448517
Epoch  300 | Loss: 8.979653
Epoch  400 | Loss: 3.121897
Epoch  500 | Loss: 1.401266
Epoch  600 | Loss: 1.958241
Epoch  700 | Loss: 1.335813
Epoch  800 | Loss: 1.440770
Epoch  900 | Loss: 1.456085
Epoch 1000 | Loss: 1.289527


# 37. Inference

During inference:

```python
model.eval()
```

and usually:

```python
with torch.no_grad():
```

or:

```python
with torch.inference_mode():
```

In [35]:
model.eval()

test_x = torch.tensor([[0.], [1.], [5.], [10.]])

with torch.no_grad():
    predictions = model(test_x)

print("x:", test_x.squeeze())
print("Predictions:", predictions.squeeze())
print("Expected:", (2 * test_x + 1).squeeze())

x: tensor([ 0.,  1.,  5., 10.])
Predictions: tensor([ 1.0596,  3.4203, 12.8410, 17.9210])
Expected: tensor([ 1.,  3., 11., 21.])


# 38. `train()` vs `eval()`

These modes matter especially for:

- Dropout
- BatchNorm

Training:

```python
model.train()
```

Evaluation:

```python
model.eval()
```

`eval()` does **not** itself disable gradient calculation.

For inference, use:

```python
model.eval()

with torch.no_grad():
    ...
```

# 39. Dropout

Dropout randomly removes activations during training.

Syntax:

```python
nn.Dropout(p=0.5)
```

`p` is the probability of dropping an activation during training.

During evaluation, dropout is disabled.

In [36]:
dropout = nn.Dropout(p=0.5)
x = torch.ones(10)

dropout.train()
print("Training:", dropout(x))

dropout.eval()
print("Evaluation:", dropout(x))

Training: tensor([2., 0., 2., 0., 0., 2., 2., 0., 0., 0.])
Evaluation: tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])


# 40. Batch Normalization

For a 2D feature tensor:

```text
(batch_size, num_features)
```

use:

```python
nn.BatchNorm1d(num_features)
```

`num_features` must match the feature dimension.

In [46]:
bn = nn.BatchNorm1d(num_features=4)

x = torch.randn(8, 4)
y = bn(x)

print("Input :", x.shape)
print("Input :", x)
print("Output:", y.shape)
print("Output:", y)

Input : torch.Size([8, 4])
Input : tensor([[-0.6637,  0.3658, -0.3992,  0.4967],
        [-2.3692, -0.2767, -0.3386, -0.8810],
        [ 1.2904, -1.1756, -0.0783, -0.9706],
        [ 1.4724, -1.3775, -1.2044, -0.4824],
        [-1.2524, -1.5844, -2.5447,  1.3719],
        [-0.5379, -0.1756, -0.3683, -0.8335],
        [ 1.3407,  0.9200, -0.3788, -1.5598],
        [-0.8010,  0.3588,  1.2862,  0.8211]])
Output: torch.Size([8, 4])
Output: tensor([[-0.3594,  0.8530,  0.1033,  0.7833],
        [-1.6535,  0.1063,  0.1635, -0.6529],
        [ 1.1234, -0.9384,  0.4219, -0.7462],
        [ 1.2615, -1.1732, -0.6963, -0.2374],
        [-0.8061, -1.4136, -2.0271,  1.6955],
        [-0.2639,  0.2238,  0.1340, -0.6033],
        [ 1.1615,  1.4972,  0.1236, -1.3604],
        [-0.4635,  0.8449,  1.7769,  1.1214]],
       grad_fn=<NativeBatchNormBackward0>)


BatchNorm uses batch statistics during training and running statistics during evaluation.

We will study BatchNorm more deeply in later notebooks.

# 41. Validation

Training data is used to update parameters.

Validation data is used to measure performance without updating parameters.

Typical validation pattern:

```python
model.eval()

with torch.no_grad():
    outputs = model(validation_inputs)
    validation_loss = loss_fn(outputs, validation_targets)
```

No `backward()` and no `optimizer.step()` should occur during ordinary validation.

# 42. Binary Classification Dataset

Let's create a small artificial dataset.

Class 0 is centered near `(-2,-2)`.

Class 1 is centered near `(2,2)`.

In [38]:
torch.manual_seed(42)

n = 200

class0 = torch.randn(n // 2, 2) + torch.tensor([-2., -2.])
class1 = torch.randn(n // 2, 2) + torch.tensor([2., 2.])

X = torch.cat([class0, class1])

y = torch.cat([
    torch.zeros(n // 2),
    torch.ones(n // 2)
]).reshape(-1, 1)

print("X:", X.shape)
print("y:", y.shape)

X: torch.Size([200, 2])
y: torch.Size([200, 1])


# 43. Binary Classifier

Architecture:

```text
2 features
 ↓
Linear(2 → 8)
 ↓
ReLU
 ↓
Linear(8 → 1)
 ↓
Logit
```

Because we use `BCEWithLogitsLoss`, the final layer produces raw logits.

In [39]:
model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

print(model)

Sequential(
  (0): Linear(in_features=2, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)


In [40]:
for epoch in range(300):
    model.train()

    optimizer.zero_grad()

    logits = model(X)

    loss = loss_fn(logits, y)

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch {epoch+1:3d} | "
            f"Loss: {loss.item():.4f}"
        )

Epoch  50 | Loss: 0.0392
Epoch 100 | Loss: 0.0113
Epoch 150 | Loss: 0.0067
Epoch 200 | Loss: 0.0045
Epoch 250 | Loss: 0.0032
Epoch 300 | Loss: 0.0023


# 44. Binary Predictions

The model outputs logits.

Convert logits to probabilities:

```python
probabilities = torch.sigmoid(logits)
```

Then use a threshold such as `0.5`.

In [41]:
model.eval()

with torch.no_grad():
    logits = model(X)
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= 0.5).float()

accuracy = (predictions == y).float().mean()

print("Accuracy:", accuracy.item())

Accuracy: 1.0


Training:

```text
logits → BCEWithLogitsLoss
```

Prediction:

```text
logits → sigmoid → probability → threshold
```

These are different stages and should not be confused.

# 45. Saving a Model

A common approach is to save the model's `state_dict`:

```python
torch.save(model.state_dict(), "model.pth")
```

The state dictionary contains learned parameters and persistent buffers.

In [42]:
path = "binary_classifier_state.pth"

torch.save(model.state_dict(), path)

print("Saved:", path)

Saved: binary_classifier_state.pth


# 46. Loading a Model

Create the same architecture and load the state dictionary.

In [43]:
new_model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

new_model.load_state_dict(
    torch.load(path, weights_only=True)
)

new_model.eval()

with torch.no_grad():
    loaded_output = new_model(X)

print("Loaded output shape:", loaded_output.shape)

Loaded output shape: torch.Size([200, 1])


# 47. Full Training Template

A useful template to remember:

```python
model = Model()

loss_fn = LossFunction()

optimizer = Optimizer(
    model.parameters(),
    lr=...
)

for epoch in range(num_epochs):

    model.train()

    optimizer.zero_grad()

    outputs = model(inputs)

    loss = loss_fn(outputs, targets)

    loss.backward()

    optimizer.step()
```

Validation:

```python
model.eval()

with torch.no_grad():
    outputs = model(validation_inputs)
    validation_loss = loss_fn(
        outputs,
        validation_targets
    )
```

# 48. Parameter Cheat Sheet

| Component | Important parameters | What they control |
|---|---|---|
| `nn.Linear` | `in_features` | Input feature count |
| `nn.Linear` | `out_features` | Output feature count |
| `nn.Linear` | `bias` | Learnable bias |
| `nn.ReLU` | `inplace` | In-place behavior |
| `nn.Softmax` | `dim` | Dimension containing classes |
| `MSELoss` | `reduction` | Mean/sum/individual losses |
| `L1Loss` | `reduction` | Mean/sum/individual losses |
| `CrossEntropyLoss` | `weight`, `reduction` | Multiclass loss behavior |
| `BCELoss` | `reduction` | Binary probability loss behavior |
| `BCEWithLogitsLoss` | `weight`, `pos_weight`, `reduction` | Binary logits loss behavior |
| `SGD` | `lr` | Step size |
| `SGD` | `momentum` | Momentum contribution |
| `SGD` | `weight_decay` | Regularization |
| `Adam` | `lr` | Learning rate |
| `Adam` | `betas` | Moment decay rates |
| `Adam` | `eps` | Numerical stability |
| `Adam` | `weight_decay` | Regularization |
| `AdamW` | `weight_decay` | Decoupled weight decay |
| `Dropout` | `p` | Drop probability |
| `BatchNorm1d` | `num_features` | Feature count |

# 49. Common Mistakes

### ❌ Wrong `in_features`

Input:

```text
(batch, 10)
```

requires:

```python
nn.Linear(10, ...)
```

### ❌ Softmax before CrossEntropyLoss

Use raw logits.

### ❌ Sigmoid before BCEWithLogitsLoss

Use raw logits.

### ❌ Forgetting `zero_grad()`

Gradients accumulate.

### ❌ Forgetting `optimizer.step()`

Parameters will not update.

### ❌ Forgetting `model.eval()`

Dropout and BatchNorm may behave differently during evaluation.

### ❌ Forgetting `no_grad()` during inference

Unnecessary gradient tracking consumes resources.

### ❌ Wrong target dtype for CrossEntropyLoss

Class indices should normally be `torch.long`.

# 51. Final Challenge 🚀 — Multiclass Classifier

Create three synthetic classes:

```text
Class 0 → (-3, -3)
Class 1 → ( 0,  3)
Class 2 → ( 3, -2)
```

Each sample has two features.

Build:

```text
2
↓
Linear(2 → 16)
↓
ReLU
↓
Linear(16 → 8)
↓
ReLU
↓
Linear(8 → 3)
```

Use:

```text
CrossEntropyLoss
Adam
learning_rate = 0.01
```

Requirements:

1. Create the dataset.
2. Create the model.
3. Print the architecture.
4. Count trainable parameters.
5. Train the model.
6. Use `model.train()`.
7. Use `optimizer.zero_grad()`.
8. Forward pass.
9. Calculate loss.
10. `backward()`.
11. `optimizer.step()`.
12. Switch to `model.eval()`.
13. Use `torch.no_grad()`.
14. Calculate accuracy.

In [44]:
torch.manual_seed(42)

n_per_class = 100

class0 = torch.randn(n_per_class, 2) + torch.tensor([-3., -3.])
class1 = torch.randn(n_per_class, 2) + torch.tensor([0., 3.])
class2 = torch.randn(n_per_class, 2) + torch.tensor([3., -2.])

X = torch.cat([class0, class1, class2])

y = torch.cat([
    torch.zeros(n_per_class, dtype=torch.long),
    torch.ones(n_per_class, dtype=torch.long),
    torch.full((n_per_class,), 2, dtype=torch.long)
])

model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 3)
)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

print(model)

total = sum(p.numel() for p in model.parameters())
print("Trainable parameters:", total)

for epoch in range(300):
    model.train()

    optimizer.zero_grad()

    logits = model(X)
    loss = loss_fn(logits, y)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch {epoch+1:3d} | "
            f"Loss: {loss.item():.4f}"
        )

model.eval()

with torch.no_grad():
    logits = model(X)
    predictions = logits.argmax(dim=1)
    accuracy = (predictions == y).float().mean()

print("Final accuracy:", accuracy.item())

Sequential(
  (0): Linear(in_features=2, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=3, bias=True)
)
Trainable parameters: 211
Epoch  50 | Loss: 0.0090
Epoch 100 | Loss: 0.0016
Epoch 150 | Loss: 0.0006
Epoch 200 | Loss: 0.0004
Epoch 250 | Loss: 0.0003
Epoch 300 | Loss: 0.0002
Final accuracy: 1.0


# 52. Knowledge Check

Before moving on, explain:

1. What is `nn.Module`?
2. Why call `super().__init__()`?
3. What does `nn.Linear` calculate?
4. What are `in_features` and `out_features`?
5. What does `bias=False` change?
6. How do you count model parameters?
7. Why do we need nonlinear activation functions?
8. What does ReLU do?
9. What range does sigmoid produce?
10. What does softmax do?
11. What does the `dim` parameter of softmax mean?
12. When is MSELoss appropriate?
13. When is CrossEntropyLoss appropriate?
14. Why does CrossEntropyLoss receive logits?
15. Why does BCEWithLogitsLoss receive logits?
16. What does an optimizer do?
17. What does learning rate control?
18. What does momentum do?
19. What does weight decay do?
20. What is the practical difference between Adam and AdamW?
21. Why call `optimizer.zero_grad()`?
22. Why call `loss.backward()`?
23. Why call `optimizer.step()`?
24. Why use `model.train()`?
25. Why use `model.eval()`?
26. Why use `torch.no_grad()` during inference?

# 53. Complete PyTorch Training Pipeline

You now have the complete basic structure:

```text
                    DATA
                      │
                      ▼
                 Input Tensor
                      │
                      ▼
                 Neural Network
                      │
                      ▼
                    Logits
                      │
                      ▼
                 Loss Function
                      │
                      ▼
                 loss.backward()
                      │
                      ▼
                   Gradients
                      │
                      ▼
               optimizer.step()
                      │
                      ▼
              Updated Parameters
                      │
                      └──── repeat
```

Evaluation:

```text
Input
  ↓
model.eval()
  ↓
no_grad / inference_mode
  ↓
Prediction
  ↓
Metric
```